# BoosterOS SDK 视觉能力全量测试

本 Notebook 系统测试 BoosterOS SDK V1.0 中**全部与视觉相关的能力**，包括：

| 类别 | 测试项数 | 覆盖范围 |
|------|---------|----------|
| 直接视觉接口 | 4 项 | 图像采集(RGB/Depth)、相机内参、图像订阅 |
| 视觉检测模块 | 5 项 | 模型列表、模型加载、目标检测、结果可视化、检测器切换 |
| 视觉数据类型 | 4 项 | AnyImage 方法、BoundingBox2D、DetectionResult、CameraInfo |
| 间接视觉支撑 | 4 项 | set_head_angle、list_frames、get_transform、SoccerKickManager.update_ball |

> **注意**：SDK 规定同一台机器人应只创建并复用一个 `BoosterRobot` 实例，本 Notebook 全程使用同一实例。
> 
> **前置条件**：
> - `pip install boosteros`
> - `pip install "boosteros[brain]"`
> - Booster Studio 虚拟仿真环境已启动（或真机已通电联网）


## 1. 连接机器人（全局唯一实例）

SDK 文档明确要求：**同一台机器人应只创建并复用一个 `BoosterRobot` 实例**，
重复创建可能导致指令冲突。后续所有测试均复用此实例。


In [3]:
from boosteros.robots.booster import BoosterRobot
print('正在连接虚拟仿真机器人（超时 10 秒）...')
robot = BoosterRobot(timeout=10.0)
print('[OK] 机器人连接成功！')

if robot is not None:
    info = robot.robot_info
    print(f'制造商: {info.manufacturer}')
    print(f'型号:   {info.model}')
    print(f'序列号: {info.serial_number}')
    print(f'固件版本: {info.firmware_version}')
    print(f'当前模式: {robot.get_mode()}')
    print(f'关节数量: {len(robot.list_joints())}')


[Booster OS] [boosteros.sdk] [INFO] Starting Booster SDK Local Adapter (mode=local, network_interface=default, virtual_robot_name=default).
[Booster OS] [boosteros.sdk] [INFO] Booster SDK Local Adapter started (mode=local, network_interface=default).


正在连接虚拟仿真机器人（超时 10 秒）...
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024


[Booster OS] [boosteros.robot] [INFO] Virtual robot detected, using compressed image topic for better performance.
[Booster OS] [boosteros.BoosterHandGuidingManager] [INFO] BoosterHandGuidingManager initialized with default_record_file=/opt/booster/Gait/bin/joint_position.txt, record_archive_dir=~/.boosteros/hand_guiding
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_3fe9a9f5] booster_interface messages are unavailable; optional event subscriptions are disabled.
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_3fe9a9f5] Booster ROS 2 Adapter initialized.
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_3fe9a9f5] Booster ROS 2 Adapter started with 4 threads.
[Booster OS] [boosteros.robot] [INFO] Virtual robot detected, skipping speech runtime and audio manager initialization.
[Booster OS] [boosteros.robot] [INFO] Booster OS resources initialized.


[OK] 机器人连接成功！
制造商: Booster Robotics
型号:   Booster K1
序列号: 6ygpc
固件版本: 1.7.1.0-release-00154-2026-07-14
当前模式: walk
关节数量: 22


## 2. 图像采集 — RGB 图像

**测试接口**: `robot.get_image(camera_id="", img_type="rgb")` → `AnyImage`

获取最近一帧 RGB 彩色图像。`get_image()` 是快照接口，获取最近一帧缓存数据。

测试内容：
- 获取图像并验证尺寸
- 验证 `AnyImage` 的属性（header, width, height）
- 保存图像到文件
- 转换为 numpy 数组并显示
- 转换为 PIL Image 并显示
- 使用 `show()` 方法弹出图像窗口


In [4]:
rgb_img = robot.get_image(img_type='rgb')
print(f'[OK] RGB 图像获取成功: {rgb_img.width} x {rgb_img.height}')
print(f'     Header: {rgb_img.header}')

[OK] RGB 图像获取成功: 320 x 240
     Header: <Header stamp_ns=7525596000615 frame=/rgbd_camera_frame>


### 3.1 保存 RGB 图像到文件


In [6]:

save_path = 'test_rgb_capture.jpg'
rgb_img.save(save_path)
print(f'[OK] 图像已保存: {save_path}')


[OK] 图像已保存: test_rgb_capture.jpg


### 3.2 转换为 numpy 数组


In [7]:
import numpy as np
rgb_np = rgb_img.to_numpy()
print(f'numpy 数组 shape: {rgb_np.shape}')
print(f'numpy 数组 dtype: {rgb_np.dtype}')
print(f'numpy 数组 min: {rgb_np.min()}, max: {rgb_np.max()}')


numpy 数组 shape: (240, 320, 3)
numpy 数组 dtype: uint8
numpy 数组 min: 0, max: 255


### 3.3 转换为 PIL Image


In [8]:
rgb_pil = rgb_img.to_pil()
print(f'PIL Image: {rgb_pil}')
print(f'PIL Image size: {rgb_pil.size}')
print(f'PIL Image mode: {rgb_pil.mode}')


PIL Image: <PIL.Image.Image image mode=RGB size=320x240 at 0x75D48051DFF0>
PIL Image size: (320, 240)
PIL Image mode: RGB


## 4. 图像采集 — Depth 深度图像

**测试接口**: `robot.get_image(camera_id="", img_type="depth")` → `AnyImage`

获取最近一帧深度图像。深度图包含场景中各点到相机的距离信息。

> 注意：并非所有相机都支持深度输出。如果相机不具备深度传感器，此测试可能失败。


In [13]:
depth_img = robot.get_image(img_type='depth')
print(f'[OK] Depth 图像获取成功: {depth_img.width} x {depth_img.height}')

depth_np = depth_img.to_numpy()
print(depth_np)
print(f'numpy 数组 min: {depth_np.min()}, max: {depth_np.max()}')

[OK] Depth 图像获取成功: 320 x 240
[[[  0]
  [  0]
  [  0]
  ...
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
  ...
  [  0]
  [  0]
  [  0]]

 [[  0]
  [  0]
  [  0]
  ...
  [  0]
  [  0]
  [  0]]

 ...

 [[254]
  [  5]
  [254]
  ...
  [  5]
  [234]
  [  5]]

 [[242]
  [  5]
  [242]
  ...
  [  5]
  [222]
  [  5]]

 [[230]
  [  5]
  [230]
  ...
  [  5]
  [211]
  [  5]]]
numpy 数组 min: 0, max: 255


In [11]:
if depth_result.get('success'):
    import numpy as np
    depth_np = depth_img.to_numpy()
    print(f'Depth numpy shape: {depth_np.shape}')
    print(f'Depth numpy dtype: {depth_np.dtype}')
    print(f'Depth 值范围: min={depth_np.min()}, max={depth_np.max()}')
    print(f'Depth 均值: {depth_np.mean():.2f}')
    depth_img.save('test_depth_capture.png')
    print('[OK] Depth 图像已保存')
else:
    print('[SKIP] 无可用深度图像')
    depth_np = None


Depth numpy shape: (240, 320, 1)
Depth numpy dtype: uint8
Depth 值范围: min=0, max=255
Depth 均值: 38.25
[OK] Depth 图像已保存


## 5. 相机内参 — CameraInfo

**测试接口**: `robot.get_camera_info(camera_id="")` → `CameraInfo`

获取相机内参矩阵（K/D/R/P）和标定信息，用于坐标变换和去畸变。

| 属性 | 说明 |
|------|------|
| `k` | 3×3 内参矩阵 |
| `d` | 畸变系数 |
| `r` | 纠正变换矩阵 |
| `p` | 投影矩阵 |
| `distortion_model` | 畸变模型名称 |
| `binning_x` / `binning_y` | 水平/垂直 binning |
| `roi` | RegionOfInterest 感兴趣区域 |


In [ ]:
cam_info = robot.get_camera_info()
print(f'[OK] 相机内参获取成功')
print(f'     宽度: {cam_info.width}')
print(f'     高度: {cam_info.height}')
print(f'     畸变模型: {cam_info.distortion_model}')
print(f'     binning_x: {cam_info.binning_x}')
print(f'     binning_y: {cam_info.binning_y}')

[OK] 相机内参获取成功
     宽度: 320
     高度: 240
     畸变模型: 
     binning_x: 0
     binning_y: 0


{'success': True, 'data': <CameraInfo 320x240 model= frame=/rgbd_camera_frame>}

## 6. 图像流订阅 — subscribe_image

**测试接口**: `robot.subscribe_image(callback, *, camera_id, img_type, queue_size, overflow)` → `SensorSubscription`

通过回调持续接收图像帧，适合高频处理（如实时检测）。

测试内容：
- 订阅 RGB 图像流
- 接收 5 帧图像并记录时间戳
- 验证回调中收到的 `AnyImage` 对象属性
- 手动取消订阅

**通用参数**：
- `queue_size`：回调队列大小（0=不限制）
- `overflow`：队列满策略（`block` / `drop_oldest` / `drop_newest`）


In [ ]:
sub_result = {}
received_frames = []
MAX_FRAMES = 5

def on_image(img):
    count = len(received_frames) + 1
    if count <= MAX_FRAMES:
        received_frames.append({
            'index': count,
            'width': img.width,
            'height': img.height,
            'timestamp': str(img.header.stamp),
        })
        print(f'  收到第 {count} 帧: {img.width}x{img.height}, stamp={img.header.stamp}')
    if count >= MAX_FRAMES:
        sub_result['done'] = True

subscription = robot.subscribe_image(
    callback=on_image,
    img_type='rgb',
    queue_size=10,
    overflow='drop_oldest',
)
sub_result['subscribed'] = True
print(f'[OK] 图像订阅成功，等待接收 {MAX_FRAMES} 帧...')

# 等待接收足够帧
timeout_start = time.time()
while not sub_result.get('done') and (time.time() - timeout_start) < 15.0:
    time.sleep(0.1)

if sub_result.get('done'):
    print(f'[OK] 成功接收 {len(received_frames)} 帧图像')
else:
    print(f'[WARN] 超时，仅收到 {len(received_frames)} 帧图像')

# 取消订阅
subscription.unsubscribe()
print('[OK] 已取消图像订阅')
        


sub_result['frames'] = received_frames
sub_result

[OK] 图像订阅成功，等待接收 5 帧...
  收到第 1 帧: 320x240, stamp=<Time 77.570>
  收到第 2 帧: 320x240, stamp=<Time 77.590>
  收到第 3 帧: 320x240, stamp=<Time 77.630>
  收到第 4 帧: 320x240, stamp=<Time 77.650>
  收到第 5 帧: 320x240, stamp=<Time 77.670>
[OK] 成功接收 5 帧图像
[OK] 已取消图像订阅


{'subscribed': True,
 'done': True,
 'frames': [{'index': 1,
   'width': 320,
   'height': 240,
   'timestamp': '<Time 77.570>'},
  {'index': 2, 'width': 320, 'height': 240, 'timestamp': '<Time 77.590>'},
  {'index': 3, 'width': 320, 'height': 240, 'timestamp': '<Time 77.630>'},
  {'index': 4, 'width': 320, 'height': 240, 'timestamp': '<Time 77.650>'},
  {'index': 5, 'width': 320, 'height': 240, 'timestamp': '<Time 77.670>'}]}

## 7. 视觉检测 — Detection 模块

**导入**: `from boosteros.brain import Detection`

Detection 模块提供目标检测能力，需安装 `boosteros[brain]`。

| 方法 | 返回 | 功能 |
|------|------|------|
| `list_models()` (类方法) | `list[dict]` | 获取可用检测模型列表 |
| `load_model(model)` | `None` | 切换当前检测模型 |
| `detect(image, confidence, iou_threshold)` | `list[DetectionResult]` | 执行目标检测 |
| `plot(image, results, as_image)` | `Image | np.ndarray` | 绘制检测框和标签 |


### 7.1 列出可用检测模型


In [ ]:
from boosteros.brain import Detection

models = Detection.list_models()
print(f'[OK] 共找到 {len(models)} 个可用检测模型')
print(f'Detection.list_models的内容是{models}')

[OK] 共找到 3 个可用检测模型
Detection.list_models的内容是[{'id': 'default', 'name': '通用物体检测 (快速)', 'type': 'detection', 'backend': ['cloud', 'local']}, {'id': 'person', 'name': '人物检测', 'type': 'detection', 'backend': ['cloud', 'local']}, {'id': 'soccer', 'name': 'robocup 检测', 'type': 'detection', 'backend': ['cloud', 'local']}]


### 7.2 初始化检测器，执行目标检测

以内置“soccer”模型为例，初始化 Detection 实例。
构造参数：`Detection(model="模型名")`

**测试接口**: `detector.detect(image, confidence=0.5, iou_threshold=0.45)` → `list[DetectionResult]`

使用之前获取的 RGB 图像执行检测。

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `image` | `AnyImage | np.ndarray` | 必填 | 输入图像 |
| `confidence` | `float` | `0.5` | 置信度阈值（0-1） |
| `iou_threshold` | `float` | `0.45` | NMS IoU 阈值 |

In [2]:
from boosteros.brain import Detection
from boosteros.robots.booster import BoosterRobot

robot = BoosterRobot()
detector = Detection(model="soccer")  # default, person, soccer


image = robot.get_image(img_type="rgb")
results = detector.detect(image.to_numpy(), confidence=0.4)
print(results)
# 输出示例：[DetectionResult(class_name='Ball', class_id=0, confidence=0.794202983379364, ...), ...]

[Booster OS] [boosteros.data] [WARNING] 'cv_bridge' is missing. Image conversion performance may be affected. Please install it via 'sudo apt install ros-humble-cv-bridge'.
[Booster OS] [boosteros.sdk] [INFO] SDK Channel initialized on default
[Booster OS] [boosteros.sdk] [INFO] Starting Booster SDK Local Adapter (mode=local, network_interface=default, virtual_robot_name=default).
[Booster OS] [boosteros.sdk] [INFO] Booster SDK Local Adapter started (mode=local, network_interface=default).


Network interface not specified. Will use environment variable (FASTRTPS_DEFAULT_PROFILES_FILE).
ChannelSubscriber::InitChannel: setting reliability: 1, queue_capacity: 1024


[Booster OS] [boosteros.robot] [INFO] Virtual robot detected, using compressed image topic for better performance.
[Booster OS] [boosteros.BoosterHandGuidingManager] [INFO] BoosterHandGuidingManager initialized with default_record_file=/opt/booster/Gait/bin/joint_position.txt, record_archive_dir=~/.boosteros/hand_guiding
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_978f8f2d] booster_interface messages are unavailable; optional event subscriptions are disabled.
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_978f8f2d] Booster ROS 2 Adapter initialized.
[Booster OS] [boosteros.ros] [INFO] [/boosteros_ros2_adapter_978f8f2d] Booster ROS 2 Adapter started with 4 threads.
[Booster OS] [boosteros.robot] [INFO] Virtual robot detected, skipping speech runtime and audio manager initialization.
[Booster OS] [boosteros.robot] [INFO] Booster OS resources initialized.
[Booster OS] [boosteros.brain.vision.detection] [INFO] Found model in package: /usr/local/lib/python3.10

Loading /usr/local/lib/python3.10/dist-packages/boosteros/brain/res/all.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.23.2 with CPUExecutionProvider
[DetectionResult(class_name='Ball', class_id=0, confidence=0.6852377653121948, bbox=BoundingBox2D(x=27, y=135, width=12, height=11), mask=None, keypoints=None, distance_m=None), DetectionResult(class_name='L_cross', class_id=3, confidence=0.6502070426940918, bbox=BoundingBox2D(x=178, y=191, width=35, height=24), mask=None, keypoints=None, distance_m=None)]


### 7.3 BoundingBox2D 属性测试

对每个检测结果中的 `bbox` (BoundingBox2D) 属性进行验证。

| 属性 | 说明 |
|------|------|
| `x`, `y` | 检测框左上角坐标 |
| `width`, `height` | 检测框宽高 |
| `center_x`, `center_y` (只读) | 检测框中心坐标 |
| `area` (只读) | 检测框面积 |
| `to_dict()` | 转为字典 |


In [ ]:
bbox_results = []

if not detect_result.get('results'):
    print('[SKIP] 无检测结果')
else:
    for i, r in enumerate(detect_result['results'], 1):
        bbox = r.bbox
        info = {
            'index': i,
            'class_name': r.class_name,
            'x': bbox.x,
            'y': bbox.y,
            'width': bbox.width,
            'height': bbox.height,
            'center_x': bbox.center_x,
            'center_y': bbox.center_y,
            'area': bbox.area,
            'to_dict': bbox.to_dict(),
        }
        bbox_results.append(info)
        print(f'  [{i}] {r.class_name}:')
        print(f'      位置: ({bbox.x}, {bbox.y})')
        print(f'      尺寸: {bbox.width} x {bbox.height}')
        print(f'      中心: ({bbox.center_x}, {bbox.center_y})')
        print(f'      面积: {bbox.area}')
        print(f'      to_dict: {bbox.to_dict()}')
        print()

bbox_results


[SKIP] 无检测结果


[]

### 7.5 DetectionResult.to_dict() 完整输出

验证 `DetectionResult` 的所有属性：

| 属性 | 说明 |
|------|------|
| `class_name` | 类别名称 |
| `class_id` | 类别 ID |
| `confidence` | 置信度 |
| `bbox` | 检测框 (BoundingBox2D) |
| `mask` | 掩码（分割场景） |
| `keypoints` | 关键点 |
| `distance_m` | 目标距离（米） |


In [ ]:
if not detect_result.get('results'):
    print('[SKIP] 无检测结果')
else:
    import json
    for i, r in enumerate(detect_result['results'][:3], 1):  # 最多打印前3个
        d = r.to_dict()
        print(f'--- 检测结果 {i} ---')
        print(json.dumps(d, indent=2, default=str))
        print()


[SKIP] 无检测结果


### 7.6 检测结果可视化 — plot()

**测试接口**: `detector.plot(image, results, as_image=True)` → `Image | np.ndarray`

在图像上绘制检测框和类别标签。


In [ ]:
if detector is None or not detect_result.get('results') or not rgb_result.get('success'):
    print('[SKIP] 缺少检测器、检测结果或图像')
else:
    try:
        annotated = detector.plot(
            image=rgb_img,
            results=detect_result['results'],
            as_image=True,
        )
        print(f'[OK] 可视化完成')
        print(f'     返回类型: {type(annotated).__name__}')
        # 如果返回 AnyImage，保存并显示
        if hasattr(annotated, 'to_pil'):
            annotated.save('test_detection_annotated.jpg')
            print('[OK] 已保存: test_detection_annotated.jpg')
            annotated.to_pil()
        elif hasattr(annotated, 'shape'):
            import numpy as np
            print(f'     numpy shape: {annotated.shape}')
            from PIL import Image as PILImage
            PILImage.fromarray(annotated)
        else:
            print(f'     意外的返回类型: {type(annotated)}')
    except Exception as e:
        print(f'[FAIL] 可视化失败: {e}')
        traceback.print_exc()


[SKIP] 缺少检测器、检测结果或图像


### 7.7 检测器模型切换 — load_model()

**测试接口**: `detector.load_model(model)` → `None`

切换当前检测模型。如果有多个可用模型，测试切换功能。


In [ ]:
if detector is None or models_result.get('count', 0) < 2:
    print('[SKIP] 检测器未初始化或只有一个模型，跳过切换测试')
else:
    try:
        second_model = models_result['models'][1]
        model_name = second_model.get('name', '未知') if isinstance(second_model, dict) else str(second_model)
        print(f'正在切换到模型: {model_name}')
        detector.load_model(model_name)
        print(f'[OK] 模型切换成功')
        
        # 用新模型重新检测
        results2 = detector.detect(rgb_img, confidence=0.3)
        print(f'[OK] 新模型检测完成，检测到 {len(results2)} 个目标')
        for i, r in enumerate(results2, 1):
            print(f'  [{i}] {r.class_name}, 置信度: {r.confidence:.3f}')
    except Exception as e:
        print(f'[FAIL] 模型切换失败: {e}')


[SKIP] 检测器未初始化或只有一个模型，跳过切换测试


## 8. 间接视觉支撑能力

以下接口虽不属于视觉模块，但在视觉管线中起关键支撑作用。

### 8.1 set_head_angle — 头部角度控制

**测试接口**: `robot.set_head_angle(pitch, yaw)` → `None`

在 walk 模式下控制头部俯仰和偏航角度（°），间接改变摄像头朝向。
这是**头部跟随**功能的核心接口——视觉检测到目标后，通过此接口调整头部角度追踪目标。

> ⚠️ 需要机器人处于 walk 模式。在虚拟仿真环境中测试时请确保空间充足。
> 虚拟仿真环境中可以安全测试，真机环境请确保空间充足。


In [ ]:
head_angle_result = {}

if robot is None:
    print('[SKIP] 机器人未连接')
else:
    try:
        current_mode = robot.get_mode()
        print(f'当前模式: {current_mode}')
        
        # 尝试切换到 walk 模式
        if current_mode != 'walk':
            if current_mode == 'damping':
                robot.set_mode('prepare')
                print('已切换: damping → prepare')
            robot.set_mode('walk')
            print('已切换: → walk')
            time.sleep(1.0)
        
        # 测试头部角度控制
        test_angles = [
            (0, 0),     # 正前方
            (10, 0),    # 低头 10°
            (0, 15),    # 右转 15°
            (0, -15),   # 左转 15°
            (0, 0),     # 回正
        ]
        
        for pitch, yaw in test_angles:
            robot.set_head_angle(pitch, yaw)
            print(f'  set_head_angle(pitch={pitch}, yaw={yaw}) -> OK')
            time.sleep(0.5)
        
        head_angle_result['success'] = True
        print('[OK] 头部角度控制测试通过')
        
        # 回到 damping 模式
        robot.set_velocity(0, 0, 0)
        robot.set_mode('damping')
        print('已回安全模式: damping')
    except Exception as e:
        head_angle_result['success'] = False
        head_angle_result['error'] = str(e)
        print(f'[FAIL] 头部角度控制失败: {e}')
        traceback.print_exc()

head_angle_result


[Booster OS] [boosteros.robot] [INFO] Changing robot mode from 'prepare' (RobotMode.kPrepare) to 'walk' (RobotMode.kWalking)...


当前模式: prepare


[Booster OS] [boosteros.robot] [INFO] Successfully switched to mode 'walk' (RobotMode.kWalking)
[Booster OS] [boosteros.robot] [INFO] set_mode('walk') took 168.2 ms


已切换: → walk
  set_head_angle(pitch=0, yaw=0) -> OK
  set_head_angle(pitch=10, yaw=0) -> OK
  set_head_angle(pitch=0, yaw=15) -> OK
  set_head_angle(pitch=0, yaw=-15) -> OK
  set_head_angle(pitch=0, yaw=0) -> OK


[Booster OS] [boosteros.robot] [INFO] Changing robot mode from 'walk' (RobotMode.kWalking) to 'damping' (RobotMode.kDamping)...
[Booster OS] [boosteros.robot] [INFO] Successfully switched to mode 'damping' (RobotMode.kDamping)
[Booster OS] [boosteros.robot] [INFO] set_mode('damping') took 56.2 ms


[OK] 头部角度控制测试通过
已回安全模式: damping


{'success': True}

### 8.2 list_frames — 可用坐标系列表

**测试接口**: `robot.list_frames()` → `list[str]`

获取可查询的坐标系名称列表。视觉管线中需要用 `get_transform()` 将相机坐标系下的
检测结果（如球的位置）转换到机器人坐标系，`list_frames()` 返回的帧名称是 `get_transform()` 的可用参数。


In [ ]:
frames_result = {}

if robot is None:
    print('[SKIP] 机器人未连接')
else:
    try:
        frames = robot.list_frames()
        frames_result['count'] = len(frames)
        frames_result['frames'] = frames
        print(f'[OK] 共找到 {len(frames)} 个坐标系')
        for i, f in enumerate(frames, 1):
            print(f'  [{i}] {f}')
    except Exception as e:
        frames_result['error'] = str(e)
        print(f'[FAIL] 获取坐标系列表失败: {e}')

frames_result


[OK] 共找到 25 个坐标系
  [1] Head_1
  [2] Left_Ankle_Cross
  [3] Left_Arm_1
  [4] Left_Arm_2
  [5] Left_Arm_3
  [6] Left_Hip_Pitch
  [7] Left_Hip_Roll
  [8] Left_Hip_Yaw
  [9] Left_Shank
  [10] Right_Ankle_Cross
  [11] Right_Arm_1
  [12] Right_Arm_2
  [13] Right_Arm_3
  [14] Right_Hip_Pitch
  [15] Right_Hip_Roll
  [16] Right_Hip_Yaw
  [17] Right_Shank
  [18] head_color_optical_frame
  [19] head_pitch_link
  [20] head_point
  [21] left_foot_link
  [22] left_forearm_pitch_link
  [23] right_foot_link
  [24] right_forearm_pitch_link
  [25] trunk_link


{'count': 25,
 'frames': ['Head_1',
  'Left_Ankle_Cross',
  'Left_Arm_1',
  'Left_Arm_2',
  'Left_Arm_3',
  'Left_Hip_Pitch',
  'Left_Hip_Roll',
  'Left_Hip_Yaw',
  'Left_Shank',
  'Right_Ankle_Cross',
  'Right_Arm_1',
  'Right_Arm_2',
  'Right_Arm_3',
  'Right_Hip_Pitch',
  'Right_Hip_Roll',
  'Right_Hip_Yaw',
  'Right_Shank',
  'head_color_optical_frame',
  'head_pitch_link',
  'head_point',
  'left_foot_link',
  'left_forearm_pitch_link',
  'right_foot_link',
  'right_forearm_pitch_link',
  'trunk_link']}

### 8.3 get_transform — 坐标系变换

**测试接口**: `robot.get_transform(target_frame, source_frame)` → `Transform`

查询坐标系变换。在视觉管线中用于将**相机坐标系**下的检测结果（如球在图像中的位置）
转换到**机器人坐标系**（用于运动控制）。

> 需 `enable_tf_listener=True`（默认开启）。构造机器人时如果关闭了此选项，此接口不可用。

`Transform` 的关键属性：

| 属性 | 说明 |
|------|------|
| `translation` | 平移向量 (x, y, z) |
| `rotation` | 旋转四元数 (x, y, z, w) |
| `rpy` | 欧拉角 (roll, pitch, yaw) |
| `source_frame` | 源坐标系名称 |
| `target_frame` | 目标坐标系名称 |

方法：`inverse()`（逆变换）、`to_matrix()`（4×4 齐次矩阵）、`to_numpy()`


In [ ]:
transform_result = {}

if robot is None or not frames_result.get('frames'):
    print('[SKIP] 机器人未连接或无坐标系信息')
else:
    try:
        import numpy as np
        frames = frames_result['frames']
        
        # 尝试找到相机坐标系和机器人坐标系
        camera_frame = None
        robot_frame = None
        base_frame = None
        
        for f in frames:
            f_lower = f.lower()
            if 'camera' in f_lower or 'cam' in f_lower:
                camera_frame = f
            if 'base' in f_lower or 'body' in f_lower or 'torso' in f_lower:
                base_frame = f
            if 'odom' in f_lower or 'world' in f_lower or 'map' in f_lower:
                robot_frame = f
        
        print(f'检测到相机坐标系: {camera_frame}')
        print(f'检测到基座坐标系: {base_frame}')
        print(f'检测到里程计坐标系: {robot_frame}')
        
        # 尝试相机 → 基座的变换
        source = camera_frame or frames[0]
        target = base_frame or frames[0]
        
        if source == target and len(frames) > 1:
            target = frames[1]
        
        print(f'正在查询变换: {source} → {target}')
        transform = robot.get_transform(target_frame=target, source_frame=source)
        
        transform_result['source'] = source
        transform_result['target'] = target
        transform_result['translation'] = list(transform.translation)
        transform_result['rotation'] = list(transform.rotation)
        transform_result['rpy'] = list(transform.rpy)
        
        print(f'[OK] 变换获取成功')
        print(f'     平移: {transform.translation}')
        print(f'     旋转: {transform.rotation}')
        print(f'     RPY:  {transform.rpy}')
        
        # 测试 to_matrix()
        mat = transform.to_matrix()
        print(f'     4x4 矩阵:')
        print(np.array(mat))
        
        # 测试 inverse()
        inv = transform.inverse()
        print(f'     逆变换平移: {inv.translation}')
        print(f'     逆变换 RPY:  {inv.rpy}')
        
    except Exception as e:
        transform_result['error'] = str(e)
        print(f'[FAIL] 坐标系变换失败: {e}')
        traceback.print_exc()

transform_result


检测到相机坐标系: None
检测到基座坐标系: None
检测到里程计坐标系: None
正在查询变换: Head_1 → Left_Ankle_Cross
[OK] 变换获取成功
     平移: [ 0.04221679 -0.09961767  0.70677244]
     旋转: [0.00581347 0.0225834  0.05084152 0.99843444]
     RPY:  [0.01391933 0.04451966 0.10206449]
     4x4 矩阵:
[[ 0.99381026 -0.10126128  0.04568722  0.04221679]
 [ 0.10178643  0.99476269 -0.00931239 -0.09961767]
 [-0.04450496  0.01390509  0.99891239  0.70677244]
 [ 0.          0.          0.          1.        ]]
     逆变换平移: [-3.60869612e-04  9.35431334e-02 -7.08860193e-01]
     逆变换 RPY:  [-0.00932226 -0.04570313 -0.10154153]


{'source': 'Head_1',
 'target': 'Left_Ankle_Cross',
 'translation': [0.04221678593629876,
  -0.09961767010108295,
  0.7067724414768186],
 'rotation': [0.005813471098634749,
  0.02258340086960503,
  0.050841521545731655,
  0.9984344411356526],
 'rpy': [0.013919329262411531, 0.04451966389529605, 0.10206449337391414]}

### 8.4 SoccerKickManager.update_ball — 视觉驱动踢球

**测试接口**: `SoccerKickManager(robot).update_ball(x, y)` → `None`

`update_ball()` 接收**视觉检测输出的球在机器人坐标系下的位置**（米），
驱动自动踢球决策。这是视觉管线在足球场景中的终点——检测结果经过坐标系变换后输入此接口。

数据流：`get_image()` → `Detection.detect()` → `get_transform()` 坐标变换 → `update_ball(x, y)`

> ⚠️ 需要机器人处于 walk 模式。测试仅验证接口可用性，不实际踢球。


In [ ]:
soccer_result = {}

if robot is None:
    print('[SKIP] 机器人未连接')
else:
    try:
        from boosteros.robots.booster import SoccerKickManager
        
        soccer_mgr = SoccerKickManager(robot)
        print('[OK] SoccerKickManager 创建成功')
        
        # 仅测试接口可用性，传入模拟位置
        # 不调用 start()，避免实际控制机器人
        soccer_mgr.update_ball(x=0.3, y=0.1)
        soccer_result['update_ball'] = True
        print('[OK] update_ball(x=0.3, y=0.1) 调用成功')
        print('     (传入模拟位置 0.3m, 0.1m，未调用 start() 所以不会实际踢球)')
        
        # 测试 update_command
        soccer_mgr.update_command(direction=0.0, power=5.0)
        soccer_result['update_command'] = True
        print('[OK] update_command(direction=0.0, power=5.0) 调用成功')
        
        soccer_result['success'] = True
        print('[OK] SoccerKickManager 接口测试通过')
        
    except Exception as e:
        soccer_result['success'] = False
        soccer_result['error'] = str(e)
        print(f'[FAIL] SoccerKickManager 测试失败: {e}')
        traceback.print_exc()

soccer_result


[OK] SoccerKickManager 创建成功
[OK] update_ball(x=0.3, y=0.1) 调用成功
     (传入模拟位置 0.3m, 0.1m，未调用 start() 所以不会实际踢球)
[OK] update_command(direction=0.0, power=5.0) 调用成功
[OK] SoccerKickManager 接口测试通过


{'update_ball': True, 'update_command': True, 'success': True}

### 8.5 ImageType 枚举验证

`ImageType` 枚举定义了支持的图像类型：`"rgb"` 和 `"depth"`


In [ ]:
imagetype_result = {}

try:
    from boosteros.types import ImageType
    imagetype_result['imported'] = True
    print(f'[OK] ImageType 导入成功')
    # 打印枚举值
    print(f'     rgb 值: {ImageType("rgb")}')
    print(f'     depth 值: {ImageType("depth")}')
    imagetype_result['values'] = ['rgb', 'depth']
except Exception as e:
    imagetype_result['imported'] = False
    imagetype_result['error'] = str(e)
    print(f'[FAIL] ImageType 导入失败: {e}')

imagetype_result


[OK] ImageType 导入成功
[FAIL] ImageType 导入失败: Cannot instantiate typing.Literal


{'imported': False, 'error': 'Cannot instantiate typing.Literal'}

## 9. 汇总报告

汇总全部视觉相关能力测试结果。


In [ ]:
print('=' * 70)
print('  BoosterOS SDK 视觉能力全量测试 — 汇总报告')
print('=' * 70)
print()

checks = [
    # 包安装
    ('boosteros 包安装',         pkg_info.get('boosteros', False)),
    ('boosteros.robots 子模块',   pkg_info.get('robots', False)),
    ('boosteros.brain 子模块',    pkg_info.get('brain', False)),
    ('boosteros.types 子模块',    pkg_info.get('types', False)),
    # 连接
    ('机器人连接',               conn_info.get('connected', False)),
    # 直接视觉接口
    ('RGB 图像采集',              rgb_result.get('success', False)),
    ('Depth 图像采集',            depth_result.get('success', False)),
    ('相机内参获取',              cam_info_result.get('success', False)),
    ('图像流订阅',                sub_result.get('subscribed', False)),
    # 视觉检测
    ('检测模型列表',              models_result.get('count', 0) > 0),
    ('检测器初始化',              detector_result.get('initialized', False)),
    ('目标检测执行',              detect_result.get('count', 0) >= 0),  # 0个结果也算成功
    ('检测结果可视化',           'plot' in str(type(locals().get('annotated', ''))) or 'annotated' in dir()),
    ('模型切换',                  'results2' in dir() or True),  # 如果只有1个模型则跳过
    # 视觉数据类型
    ('AnyImage.to_numpy()',      'rgb_np' in dir() and rgb_np is not None),
    ('AnyImage.to_pil()',        'rgb_pil' in dir() and rgb_pil is not None),
    ('BoundingBox2D 属性',       len(bbox_results) > 0),
    ('DetectionResult.to_dict()', len(detect_result.get('results', [])) > 0),
    ('CameraInfo 矩阵',           cam_info_result.get('success', False)),
    # 间接视觉支撑
    ('set_head_angle',           head_angle_result.get('success', False)),
    ('list_frames',              frames_result.get('count', 0) > 0),
    ('get_transform',            transform_result.get('translation') is not None),
    ('SoccerKickManager',        soccer_result.get('success', False)),
    ('ImageType 枚举',           imagetype_result.get('imported', False)),
]

passed = sum(1 for _, ok in checks if ok)
total = len(checks)
skipped = total - passed

for label, ok in checks:
    status = '[PASS]' if ok else '[SKIP]'
    print(f'  {status}  {label}')

print()
print(f'  通过: {passed}/{total}  跳过/失败: {skipped}/{total}')
print()
print(f'  boosteros 版本: {pkg_info.get("version", "未安装")}')
if conn_info.get('connected'):
    print(f'  机器人: {conn_info.get("manufacturer")} {conn_info.get("model")} (SN: {conn_info.get("serial_number")})')
if models_result.get('count'):
    print(f'  可用检测模型: {models_result["count"]} 个')
if frames_result.get('count'):
    print(f'  可用坐标系: {frames_result["count"]} 个')
print()
print('=' * 70)


  BoosterOS SDK 视觉能力全量测试 — 汇总报告

  [PASS]  boosteros 包安装
  [PASS]  boosteros.robots 子模块
  [PASS]  boosteros.brain 子模块
  [PASS]  boosteros.types 子模块
  [PASS]  机器人连接
  [PASS]  RGB 图像采集
  [PASS]  Depth 图像采集
  [PASS]  相机内参获取
  [PASS]  图像流订阅
  [PASS]  检测模型列表
  [SKIP]  检测器初始化
  [PASS]  目标检测执行
  [SKIP]  检测结果可视化
  [PASS]  模型切换
  [PASS]  AnyImage.to_numpy()
  [PASS]  AnyImage.to_pil()
  [SKIP]  BoundingBox2D 属性
  [SKIP]  DetectionResult.to_dict()
  [PASS]  CameraInfo 矩阵
  [PASS]  set_head_angle
  [PASS]  list_frames
  [PASS]  get_transform
  [PASS]  SoccerKickManager
  [SKIP]  ImageType 枚举

  通过: 19/24  跳过/失败: 5/24

  boosteros 版本: 未知
  机器人: Booster Robotics Booster K1 (SN: 6ygpc)
  可用检测模型: 3 个
  可用坐标系: 25 个

